# Install Dependencies

In [ ]:
!pip install -q sacrebleu
!pip install -q rouge-score
!pip install -q bert-score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.1 MB/s eta 0:00:00
 

In [ ]:
!pip install git+https://github.com/huggingface/evaluate@32546aafec25cdc2a5d7dd9f941fc5be56ba122f

  Cloning https://github.com/huggingface/evaluate (to revision 32546aafec25cdc2a5d7dd9f941fc5be56ba122f) to /tmp/pip-req-build-fzfmvcmx
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/evaluate /tmp/pip-req-build-fzfmvcmx
  Running command git rev-parse -q --verify 'sha^32546aafec25cdc2a5d7dd9f941fc5be56ba122f'
  Running command git fetch -q https://github.com/huggingface/evaluate 32546aafec25cdc2a5d7dd9f941fc5be56ba122f
  Resolved https://github.com/huggingface/evaluate to commit 32546aafec25cdc2a5d7dd9f941fc5be56ba122f
  Preparing metadata (setup.py) ... done
  Created wheel for evaluate: filename=evaluate-0.4.5.dev0-py3-none-any.whl size=84157 sha256=b4834957327de91c97b40d686004e8a1f3b4fe3157cfe6b06ad4fc4a220f83e6
  Stored in directory: /root/.cache/pip/wheels/89/96/43/3d812abd89a5bdb5c4e6156d219fdb043db5fb31137b368aee
Successfully built evaluate


# Import Required Modules

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import re

def normalize_arabic(text):
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[ًٌٍَُِّْ]", "", text)  # Remove short vowels (diacritics)
    text = re.sub(r"[^\w\s]", "", text)    # Remove punctuation (optional)
    return text.strip()

# Zero Shot

In [ ]:
df = pd.read_excel('summarization_data.xlsx')
pred_zero = pd.read_excel('Jais-TextSummarization-ZeroShot.xlsx')

In [ ]:
pred_zero['summary'] = df['summary']

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import evaluate
from transformers import AutoTokenizer

# Initialize storage
bleu_scores = []
bleu_scores2 = []
rougeL_p = []
rougeL_r = []
rougeL_f = []
bertscore_p = []
bertscore_r = []
bertscore_f = []

# Initialize scorer
model_name = 'aubmindlab/bert-base-arabertv2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
rouge = rouge_scorer.RougeScorer(['rougeL'], tokenizer=tokenizer)

# Iterate through rows
for _, row in pred_zero.iterrows():
    raw_references = [row['summary']]  # One reference as list of strings
    raw_predictions = [row['Generated Summary']]  # One prediction

    # Normalize Arabic
    hyp = [normalize_arabic(text) for text in raw_predictions]
    ref = [[normalize_arabic(ref) for ref in raw_references]]  # Nested list for multiple refs per prediction

    # BLEU
    bleu = sacrebleu.corpus_bleu(hyp, ref)
    bleu_scores.append(bleu.score)

    bleu2 = evaluate.load("bleu")
    results = bleu2.compute(predictions= hyp, references= ref)
    bleu_scores2.append(results['bleu'])

    # ROUGE-L
    r_scores = rouge.score(row['summary'], row['Generated Summary'])
    rougeL_p.append(r_scores['rougeL'].precision)
    rougeL_r.append(r_scores['rougeL'].recall)
    rougeL_f.append(r_scores['rougeL'].fmeasure)

# BERTScore
P, R, F = bert_score(pred_zero['Generated Summary'].tolist(), pred_zero['summary'].tolist(), lang="ar", model_type="bert-base-multilingual-cased", verbose=False)
bertscore_p = P.tolist()
bertscore_r = R.tolist()
bertscore_f = F.tolist()

# Create summary DataFrame
metrics_df = pd.DataFrame({
    "BLEU1": bleu_scores,
    "BLEU2": bleu_scores2,
    "ROUGE_L_P": rougeL_p,
    "ROUGE_L_R": rougeL_r,
    "ROUGE_L_F": rougeL_f,
    "BERT_P": bertscore_p,
    "BERT_R": bertscore_r,
    "BERT_F": bertscore_f,
})

# Calculate min, max, mean
summary_stats = metrics_df.agg(['min', 'max', 'mean'])

print(summary_stats)


          BLEU1     BLEU2  ROUGE_L_P  ROUGE_L_R  ROUGE_L_F    BERT_P  \
min    0.000000  0.000000   0.041916   0.027778   0.037736  0.609497   
max   16.739926  0.167399   0.269231   0.650000   0.323077  0.774956   
mean   1.157833  0.002710   0.143631   0.142287   0.116042  0.657656   

        BERT_R    BERT_F  
min   0.582318  0.599810  
max   0.805997  0.780071  
mean  0.662725  0.659514  


# Fewshot

In [ ]:
df = pd.read_excel('summarization_data.xlsx')
pred_few = pd.read_excel('Jais-TextSummarization-FewShot.xlsx')

In [ ]:
pred_few['summary'] = df['summary']

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import evaluate
from transformers import AutoTokenizer

# Initialize storage
bleu_scores = []
bleu_scores2 = []
rougeL_p = []
rougeL_r = []
rougeL_f = []
bertscore_p = []
bertscore_r = []
bertscore_f = []

# Initialize scorer
model_name = 'aubmindlab/bert-base-arabertv2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
rouge = rouge_scorer.RougeScorer(['rougeL'], tokenizer=tokenizer)

# Iterate through rows
for _, row in pred_few.iterrows():
    raw_references = [row['summary']]  # One reference as list of strings
    raw_predictions = [row['Generated Summary']]  # One prediction

    # Normalize Arabic
    hyp = [normalize_arabic(text) for text in raw_predictions]
    ref = [[normalize_arabic(ref) for ref in raw_references]]  # Nested list for multiple refs per prediction

    # BLEU
    bleu = sacrebleu.corpus_bleu(hyp, ref)
    bleu_scores.append(bleu.score)

    bleu2 = evaluate.load("bleu")
    results = bleu2.compute(predictions= hyp, references= ref)
    bleu_scores2.append(results['bleu'])

    # ROUGE-L
    r_scores = rouge.score(row['summary'], row['Generated Summary'])
    rougeL_p.append(r_scores['rougeL'].precision)
    rougeL_r.append(r_scores['rougeL'].recall)
    rougeL_f.append(r_scores['rougeL'].fmeasure)

# BERTScore
P, R, F = bert_score(pred_few['Generated Summary'].tolist(), pred_few['summary'].tolist(), lang="ar", model_type="bert-base-multilingual-cased", verbose=False)
bertscore_p = P.tolist()
bertscore_r = R.tolist()
bertscore_f = F.tolist()

# Create summary DataFrame
metrics_df = pd.DataFrame({
    "BLEU1": bleu_scores,
    "BLEU2": bleu_scores2,
    "ROUGE_L_P": rougeL_p,
    "ROUGE_L_R": rougeL_r,
    "ROUGE_L_F": rougeL_f,
    "BERT_P": bertscore_p,
    "BERT_R": bertscore_r,
    "BERT_F": bertscore_f,
})

# Calculate min, max, mean
summary_stats = metrics_df.agg(['min', 'max', 'mean'])

print(summary_stats)

         BLEU1  BLEU2  ROUGE_L_P  ROUGE_L_R  ROUGE_L_F    BERT_P    BERT_R  \
min   0.000000    0.0   0.058824   0.024390   0.034483  0.600312  0.561591   
max   3.817681    0.0   0.235294   0.150000   0.162162  0.683527  0.686252   
mean  0.345959    0.0   0.156471   0.067659   0.092652  0.645912  0.622614   

        BERT_F  
min   0.585002  
max   0.684887  
mean  0.633959  


# CoT

In [ ]:
df = pd.read_excel('summarization_data.xlsx')
pred_cot = pd.read_excel('Jais-TextSummarization-CoT.xlsx')

In [ ]:
pred_cot['summary'] = df['summary']

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import evaluate
from transformers import AutoTokenizer

# Initialize storage
bleu_scores = []
bleu_scores2 = []
rougeL_p = []
rougeL_r = []
rougeL_f = []
bertscore_p = []
bertscore_r = []
bertscore_f = []

# Initialize scorer
model_name = 'aubmindlab/bert-base-arabertv2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
rouge = rouge_scorer.RougeScorer(['rougeL'], tokenizer=tokenizer)

# Iterate through rows
for _, row in pred_cot.iterrows():
    raw_references = [row['summary']]  # One reference as list of strings
    raw_predictions = [row['Generated Summary']]  # One prediction

    # Normalize Arabic
    hyp = [normalize_arabic(text) for text in raw_predictions]
    ref = [[normalize_arabic(ref) for ref in raw_references]]  # Nested list for multiple refs per prediction

    # BLEU
    bleu = sacrebleu.corpus_bleu(hyp, ref)
    bleu_scores.append(bleu.score)

    bleu2 = evaluate.load("bleu")
    results = bleu2.compute(predictions= hyp, references= ref)
    bleu_scores2.append(results['bleu'])

    # ROUGE-L
    r_scores = rouge.score(row['summary'], row['Generated Summary'])
    rougeL_p.append(r_scores['rougeL'].precision)
    rougeL_r.append(r_scores['rougeL'].recall)
    rougeL_f.append(r_scores['rougeL'].fmeasure)

# BERTScore
P, R, F = bert_score(pred_cot['Generated Summary'].tolist(), pred_cot['summary'].tolist(), lang="ar", model_type="bert-base-multilingual-cased", verbose=False)
bertscore_p = P.tolist()
bertscore_r = R.tolist()
bertscore_f = F.tolist()

# Create summary DataFrame
metrics_df = pd.DataFrame({
    "BLEU1": bleu_scores,
    "BLEU2": bleu_scores2,
    "ROUGE_L_P": rougeL_p,
    "ROUGE_L_R": rougeL_r,
    "ROUGE_L_F": rougeL_f,
    "BERT_P": bertscore_p,
    "BERT_R": bertscore_r,
    "BERT_F": bertscore_f,
})

# Calculate min, max, mean
summary_stats = metrics_df.agg(['min', 'max', 'mean'])

print(summary_stats)


         BLEU1  BLEU2  ROUGE_L_P  ROUGE_L_R  ROUGE_L_F    BERT_P    BERT_R  \
min   0.000000    0.0   0.058824   0.024390   0.034483  0.600312  0.561591   
max   3.817681    0.0   0.235294   0.150000   0.162162  0.683527  0.686252   
mean  0.345959    0.0   0.156471   0.067659   0.092652  0.645912  0.622614   

        BERT_F  
min   0.585002  
max   0.684887  
mean  0.633959  
